# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
print(spaceship.shape)

(8693, 14)


**Check for data types**

In [4]:
print(spaceship.dtypes)

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object


**Check for missing values**

In [6]:
missing_values = spaceship.isnull().sum()

# Print the results
print(missing_values)

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [7]:
df = spaceship.dropna()


print(df.shape)

(6606, 14)


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [8]:
df['Cabin_Section'] = df['Cabin'].str[0]

# Verify the transformation
print(df['Cabin_Section'].unique())

['B' 'F' 'A' 'G' 'E' 'C' 'D' 'T']


C:\Users\User\AppData\Local\Temp\ipykernel_6864\3297334779.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Cabin_Section'] = df['Cabin'].str[0]


- Drop PassengerId and Name

In [9]:
df = df.drop(columns=['PassengerId', 'Name'])


print(df.head())

  HomePlanet CryoSleep  Cabin  Destination   Age    VIP  RoomService  \
0     Europa     False  B/0/P  TRAPPIST-1e  39.0  False          0.0   
1      Earth     False  F/0/S  TRAPPIST-1e  24.0  False        109.0   
2     Europa     False  A/0/S  TRAPPIST-1e  58.0   True         43.0   
3     Europa     False  A/0/S  TRAPPIST-1e  33.0  False          0.0   
4      Earth     False  F/1/S  TRAPPIST-1e  16.0  False        303.0   

   FoodCourt  ShoppingMall     Spa  VRDeck  Transported Cabin_Section  
0        0.0           0.0     0.0     0.0        False             B  
1        9.0          25.0   549.0    44.0         True             F  
2     3576.0           0.0  6715.0    49.0        False             A  
3     1283.0         371.0  3329.0   193.0        False             A  
4       70.0         151.0   565.0     2.0         True             F  


- For non-numerical columns, do dummies.

In [10]:
categorical_columns = df.select_dtypes(include=['object', 'bool']).columns

# Perform one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

# Verify the encoding
print(df_encoded.head())

    Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck  \
0  39.0          0.0        0.0           0.0     0.0     0.0   
1  24.0        109.0        9.0          25.0   549.0    44.0   
2  58.0         43.0     3576.0           0.0  6715.0    49.0   
3  33.0          0.0     1283.0         371.0  3329.0   193.0   
4  16.0        303.0       70.0         151.0   565.0     2.0   

   HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True  Cabin_A/0/S  ...  \
0               True            False           False        False  ...   
1              False            False           False        False  ...   
2               True            False           False         True  ...   
3               True            False           False         True  ...   
4              False            False           False        False  ...   

   Destination_TRAPPIST-1e  VIP_True  Transported_True  Cabin_Section_B  \
0                     True     False             False             True   
1       

**Perform Train Test Split**

In [12]:
from sklearn.model_selection import train_test_split

# Define the features (all columns except 'Transported') and target
features = df_encoded.drop(columns=['Transported_True'])
target = df_encoded['Transported_True']

# Perform the train-test split
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Check the shapes to verify the split
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5284, 5323)
X_test shape: (1322, 5323)
y_train shape: (5284,)
y_test shape: (1322,)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [14]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler


# Step 1: Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Initialize the KNN Regressor
knn_regressor = KNeighborsRegressor()

# Step 3: Fit the model
knn_regressor.fit(X_train_scaled, y_train)

print("KNN Regressor is initialized and trained.")

KNN Regressor is initialized and trained.


- Evaluate your model's performance. Comment it

In [15]:
from sklearn.metrics import mean_squared_error, r2_score

# Step 1: Make predictions using the KNN regressor
y_pred = knn_regressor.predict(X_test_scaled)

# Step 2: Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

# Step 3: Output the results
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R-squared (R²): {r2}")

Mean Squared Error (MSE): 0.2383661119515885
Root Mean Squared Error (RMSE): 0.4882275206823029
R-squared (R²): 0.046395868670983775


c:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


MSE of approximately 0.2384. This represents the average squared difference between the predicted and actual values
RMSE is about 0.4882, which provides an average prediction error in the same units as the target variable.
The R² value of approximately 0.0464 indicates that your model explains about 4.64% of the variance in the target variable. This is quite low.